# Review II: selective test-time normalization

This notebook implements the method described in the Review II report: an MLP encoder and logistic-regression head on digits, two actions (SOURCE or NORMALIZE), and gates that decide between them from an unlabeled support batch.

**Scope, please read before citing anything here:**

- This is a **fresh implementation written on 31 October 2026 from the report's description**. It is not the code that produced the Review II five-seed numbers. That code and its dataset, model and split details were on another system and are not in this project.
- So its results **are not expected to match** the reported 77.9 ± 0.9% table, and they do not confirm or refute it.
- These are custom shifts on 8 × 8 digits, not CIFAR-C, ImageNet-C, DomainNet or WILDS.
- Details the report leaves open are marked **Implementation choice**. The episode counts here are smaller than the earlier diagnostic's, and the seeds are new.

Run with uv, from the project directory:

`uv run --no-project --with numpy --with scipy --with scikit-learn --with pandas --with nbconvert --with ipykernel jupyter nbconvert --to notebook --execute --inplace review2_three_feature_gate.ipynb`

In [1]:
import os
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[name] = "1"

import platform
import warnings
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.ndimage import gaussian_filter, shift
from scipy.special import softmax
from sklearn.datasets import load_digits
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeRegressor, export_text

SEEDS = [241023, 241024, 241025, 241026, 241027]  # Implementation choice; the external seeds are unknown.
EPS = 1e-3
DEV_FAMILIES = ("clean", "noise", "brightness", "blur", "translation")
HELDOUT_FAMILIES = ("contrast", "occlusion")
SEVERITIES = (0, 1)
SUPPORT_SIZES = (16, 32)
QUERY_SIZE = 32
PRIORS = ("balanced", "dominant")
TRIALS = dict(aux=8, cal=4, test=8)  # Implementation choice: repeats per condition.
MARGINS = (0.0, 0.01, 0.02, 0.05, 0.1)  # Implementation choice: calibration grid.
FEATURES = ["concentration", "entropy", "displacement"]

print("python", platform.python_version(), "numpy", np.__version__, "scipy", scipy.__version__, "sklearn", sklearn.__version__)

python 3.12.11 numpy 2.5.4 scipy 1.18.1 sklearn 1.9.1


## 1. Data and splits

Digits has 1,797 images of 8 × 8 pixels, divided by 16 to lie in [0, 1]. As in the report, the split is 900 source, 400 auxiliary, 200 calibration and 297 evaluation images. Each original image belongs to one split only.

In [2]:
digits = load_digits()
X, y = digits.data.astype(np.float64) / 16.0, digits.target.astype(int)

def make_splits(seed):
    ids = np.arange(len(y))
    source, rest = train_test_split(ids, train_size=900, stratify=y, random_state=seed)
    aux, rest = train_test_split(rest, train_size=400, stratify=y[rest], random_state=seed + 1)
    cal, test = train_test_split(rest, train_size=200, stratify=y[rest], random_state=seed + 2)
    return dict(source=source, aux=aux, cal=cal, test=test)

{k: len(v) for k, v in make_splits(SEEDS[0]).items()}

{'source': 900, 'aux': 400, 'cal': 200, 'test': 297}

## 2. Source model

**Encoder:** one 32-unit ReLU hidden layer, trained with L-BFGS. **Head:** logistic regression on hidden features standardized with source means and variances. Both are frozen afterwards. Convergence warnings are counted rather than hidden.

In [3]:
def fit_source(X_src, y_src, seed):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        encoder = MLPClassifier(hidden_layer_sizes=(32,), activation="relu", solver="lbfgs",
                                alpha=0.01, max_iter=250, tol=1e-5, random_state=seed).fit(X_src, y_src)
        W, b = encoder.coefs_[0], encoder.intercepts_[0]
        h = np.maximum(X_src @ W + b, 0.0)
        mean, var = h.mean(0), h.var(0)
        head = LogisticRegression(C=1.0, max_iter=500, random_state=seed).fit((h - mean) / np.sqrt(var + EPS), y_src)
    n_warn = sum(issubclass(w.category, ConvergenceWarning) for w in caught)
    return dict(W=W, b=b, mean=mean, var=var, head_W=head.coef_.T, head_b=head.intercept_, convergence_warnings=n_warn)

def hidden(model, x):
    return np.maximum(x @ model["W"] + model["b"], 0.0)

def predict_with_stats(model, x, mean, var):
    z = (hidden(model, x) - mean) / np.sqrt(var + EPS)
    return softmax(z @ model["head_W"] + model["head_b"], axis=1)

## 3. Shifts

Clean, noise, brightness, blur and translation are used for development. **Contrast and occlusion are held out**: the gates never see them while being fitted or calibrated. Each has two severities.

In [4]:
def corrupt(x, family, severity, rng):
    im = x.reshape(-1, 8, 8).copy()
    if family == "noise":
        im += rng.normal(0, (0.15, 0.35)[severity], size=im.shape)
    elif family == "brightness":
        im -= (0.18, 0.35)[severity]
    elif family == "blur":
        s = (0.5, 0.9)[severity]
        im = gaussian_filter(im, sigma=(0, s, s), mode="nearest")
    elif family == "translation":
        im = shift(im, shift=(0, 1, severity), order=1, mode="constant", cval=0)
    elif family == "contrast":
        im = (im - 0.5) * (0.6, 0.3)[severity] + 0.5
    elif family == "occlusion":
        side = (2, 3)[severity]
        for one in im:
            top, left = rng.integers(1, 8 - side, size=2)
            one[top:top + side, left:left + side] = 0
    return np.clip(im, 0, 1).reshape(-1, 64)

## 4. Episodes, actions and the three signals

An episode has an unlabeled **support** batch and a separate **query** batch from disjoint images. Class mix is balanced, or about 75% one class.

- **SOURCE** scores the query with stored source statistics.
- **NORMALIZE** computes hidden-layer means and variances from support, then freezes them for the query. No weights change.

Both start from the same source state, so `delta = error_SOURCE - error_NORMALIZE`, and positive means the update helped.

The signals are computed from the support batch and source model only, never from labels:

- **Concentration:** share of the most frequently predicted class.
- **Entropy:** mean uncertainty of the predictions.
- **Displacement:** `d(B) = mean_j |mu_B,j - mu_S,j| / sqrt(var_S,j + eps)`, how far support features sit from source statistics.

**Implementation choice:** a repeated draw is allowed only when a class has too few images, as in the earlier diagnostic.

In [5]:
def draw(pool, count, prior, dominant, rng):
    if prior == "balanced":
        wanted = np.tile(np.arange(10), int(np.ceil(count / 10)))[:count]
    else:
        major = int(round(0.75 * count))
        others = np.array([c for c in range(10) if c != dominant])
        wanted = np.r_[np.full(major, dominant), rng.choice(others, count - major)]
    picked = []
    for label in range(10):
        n = int((wanted == label).sum())
        if n:
            candidates = pool[y[pool] == label]
            picked.extend(rng.choice(candidates, size=n, replace=n > len(candidates)))
    picked = np.array(picked)
    rng.shuffle(picked)
    return picked

def signals(model, support):
    p = predict_with_stats(model, support, model["mean"], model["var"])
    conc = np.bincount(p.argmax(1), minlength=10).max() / len(p)
    ent = -(p * np.log(np.clip(p, 1e-300, None))).sum(1).mean()
    disp = np.mean(np.abs(hidden(model, support).mean(0) - model["mean"]) / np.sqrt(model["var"] + EPS))
    return dict(concentration=conc, entropy=ent, displacement=disp)

def episode(model, pool, family, severity, n_support, prior, rng):
    pool = rng.permutation(pool)
    half = len(pool) // 2
    sup_pool, qry_pool = pool[:half], pool[half:]
    dominant = int(rng.integers(10))
    # With a dominant prior, support and query share the same class mix.
    s_ids, q_ids = draw(sup_pool, n_support, prior, dominant, rng), draw(qry_pool, QUERY_SIZE, prior, dominant, rng)
    support = corrupt(X[s_ids], family, severity, rng)
    query = corrupt(X[q_ids], family, severity, rng)
    h = hidden(model, support)
    acc_source = (predict_with_stats(model, query, model["mean"], model["var"]).argmax(1) == y[q_ids]).mean()
    acc_norm = (predict_with_stats(model, query, h.mean(0), h.var(0)).argmax(1) == y[q_ids]).mean()
    return dict(family=family, severity=severity, n_support=n_support, prior=prior,
                acc_source=acc_source, acc_norm=acc_norm, delta=acc_norm - acc_source, **signals(model, support))

def build(model, pool, families, trials, rng):
    return pd.DataFrame([episode(model, pool, f, s, n, p, rng)
                         for f in families for s in SEVERITIES for n in SUPPORT_SIZES for p in PRIORS for _ in range(trials)])

## 5. Gates

Each gate is a **depth-two regression tree** (at most four leaves) that predicts `delta`. NORMALIZE is chosen only if the prediction exceeds a margin. The tree is fitted on auxiliary episodes, and the margin is chosen on separate calibration episodes.

**Implementation choice:** the concentration-only gate uses the same tree on one feature, so the comparison changes only the evidence. The margin maximizes calibration accuracy, with ties going to the larger, more cautious margin.

In [6]:
def fit_gate(aux, cal, features):
    tree = DecisionTreeRegressor(max_depth=2, random_state=0).fit(aux[features], aux["delta"])
    pred = tree.predict(cal[features])
    scores = {m: np.where(pred > m, cal["acc_norm"], cal["acc_source"]).mean() for m in MARGINS}
    best = max(scores.values())
    return tree, max(m for m, v in scores.items() if v >= best - 1e-12)

def evaluate(acc, update, source):
    return dict(accuracy=100 * acc.mean(),
                harmful=100 * (update & (acc < source)).mean() if update.any() else 0.0,
                update_rate=100 * update.mean())

rows, per_family, trees, warn_counts = [], [], {}, {}
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    sp = make_splits(seed)
    model = fit_source(X[sp["source"]], y[sp["source"]], seed)
    warn_counts[seed] = model["convergence_warnings"]
    aux = build(model, sp["aux"], DEV_FAMILIES, TRIALS["aux"], rng)
    cal = build(model, sp["cal"], DEV_FAMILIES, TRIALS["cal"], rng)
    test = build(model, sp["test"], DEV_FAMILIES + HELDOUT_FAMILIES, TRIALS["test"], rng)

    decisions = {"SOURCE": np.zeros(len(test), bool), "Always normalize": np.ones(len(test), bool)}
    for name, feats in {"Concentration-only gate": ["concentration"], "Three-feature gate": FEATURES}.items():
        tree, margin = fit_gate(aux, cal, feats)
        decisions[name] = tree.predict(test[feats]) > margin
        trees[(seed, name)] = (tree, margin, feats)

    heldout = test["family"].isin(HELDOUT_FAMILIES).values
    for name, update in decisions.items():
        acc = np.where(update, test["acc_norm"], test["acc_source"])
        for scope, mask in {"all families": np.ones(len(test), bool), "held-out families": heldout}.items():
            rows.append(dict(seed=seed, policy=name, scope=scope, **evaluate(acc[mask], update[mask], test["acc_source"].values[mask])))
        frame = test.assign(acc=acc, update=update, harm=update & (acc < test["acc_source"].values), policy=name)
        per_family.append(frame.groupby(["policy", "family", "prior"], as_index=False)[["acc", "harm", "update"]].mean().assign(seed=seed))

results = pd.DataFrame(rows)
print("episodes per seed: aux", len(aux), "cal", len(cal), "test", len(test))
print("convergence warnings per seed:", warn_counts)

episodes per seed: aux 320 cal 160 test 448
convergence warnings per seed: {241023: 1, 241024: 1, 241025: 1, 241026: 1, 241027: 0}


## 6. Results of this notebook

**Accuracy** is the percentage of correct query predictions. **Harmful** is the percentage of *all* episodes in which a policy normalized and did worse than SOURCE, the same definition as the report. **±** is the standard deviation across five seeds, which shows variability and is not a confidence interval.

In [7]:
ORDER = ["SOURCE", "Always normalize", "Concentration-only gate", "Three-feature gate"]
def summary(scope):
    r = results[results["scope"] == scope]
    source = r[r.policy == "SOURCE"].set_index("seed")["accuracy"]
    out = []
    for name in ORDER:
        g = r[r.policy == name].set_index("seed")
        gain = g["accuracy"] - source
        out.append({"Policy": name, "Accuracy %": f"{g.accuracy.mean():.1f} ± {g.accuracy.std(ddof=1):.1f}",
                    "Gain vs SOURCE (pp)": f"{gain.mean():+.2f} ± {gain.std(ddof=1):.2f}",
                    "Harmful %": f"{g.harmful.mean():.1f}", "Updates %": f"{g.update_rate.mean():.1f}"})
    return pd.DataFrame(out)

display(summary("all families"))
display(summary("held-out families"))

,Policy,Accuracy %,Gain vs SOURCE (pp),Harmful %,Updates %
0,SOURCE,78.2 ± 2.8,+0.00 ± 0.00,0.0,0.0
1,Always normalize,62.2 ± 1.5,-16.00 ± 2.84,54.8,100.0
2,Concentration-only gate,79.7 ± 2.2,+1.54 ± 0.97,13.2,51.2
3,Three-feature gate,79.2 ± 2.2,+0.99 ± 1.24,9.9,37.5


,Policy,Accuracy %,Gain vs SOURCE (pp),Harmful %,Updates %
0,SOURCE,75.2 ± 4.1,+0.00 ± 0.00,0.0,0.0
1,Always normalize,64.2 ± 1.5,-11.05 ± 3.57,49.7,100.0
2,Concentration-only gate,79.1 ± 3.1,+3.90 ± 1.33,13.0,57.7
3,Three-feature gate,77.8 ± 2.6,+2.55 ± 2.74,9.7,39.2


## 7. Reference: the numbers reported in Review II

These are copied from the report for comparison. They came from another system, and **this notebook did not produce them**.

In [8]:
pd.DataFrame({"Policy": ORDER, "Reported accuracy %": ["76.4 ± 1.0", "74.8 ± 1.8", "77.3 ± 1.1", "77.9 ± 0.9"],
              "Reported gain (pp)": ["—", "-1.6", "+0.9", "+1.5"], "Reported harmful %": ["—", "31", "15", "11"]})

,Policy,Reported accuracy %,Reported gain (pp),Reported harmful %
0,SOURCE,76.4 ± 1.0,—,—
1,Always normalize,74.8 ± 1.8,-1.6,31
2,Concentration-only gate,77.3 ± 1.1,+0.9,15
3,Three-feature gate,77.9 ± 0.9,+1.5,11


## 8. Paired seed differences

Because all policies share episodes within a seed, per-seed differences are more informative than comparing means. Five seeds is few, and episodes reuse a small image pool, so read these as descriptive.

In [9]:
pivot = results[results.scope == "held-out families"].pivot(index="seed", columns="policy", values="accuracy")
paired = pd.DataFrame({"Three-feature minus concentration": pivot["Three-feature gate"] - pivot["Concentration-only gate"],
                       "Three-feature minus SOURCE": pivot["Three-feature gate"] - pivot["SOURCE"]}).round(2)
display(paired)
paired.agg(["mean", "std"]).round(2)

,Three-feature minus concentration,Three-feature minus SOURCE
seed,,
241023,-3.25,-0.24
241024,-0.54,4.39
241025,0.00,5.69
241026,0.00,3.27
241027,-2.98,-0.37


,Three-feature minus concentration,Three-feature minus SOURCE
mean,-1.35,2.55
std,1.63,2.74


## 9. Where do harmful updates come from?

Report §9 lists this breakdown as open work. This is for the implementation here only.

In [10]:
pf = pd.concat(per_family)
view = pf[pf.policy == "Three-feature gate"].groupby(["family", "prior"])[["acc", "harm", "update"]].mean().mul(100).round(1)
view.columns = ["Accuracy %", "Harmful %", "Updates %"]
view

Accuracy %  Harmful %  Updates %
family      prior                                     
blur        balanced        93.7        7.5       60.0
            dominant        88.6        4.4        6.2
brightness  balanced        93.9        4.4       60.0
            dominant        90.4        5.0        6.2
clean       balanced        97.1       18.1       60.0
            dominant        97.2        0.0        0.0
contrast    balanced        90.9        1.2       59.4
            dominant        77.6        5.0       12.5
noise       balanced        87.3       21.2       60.0
            dominant        85.2        6.2        6.2
occlusion   balanced        74.8       11.2       60.0
            dominant        67.8       21.2       25.0
translation balanced        36.0       12.5       69.4
            dominant        28.4       20.0       40.0

## 10. What the gate learned

The fitted tree for the first seed. The thresholds are specific to this model and split, and are not general rules.

In [11]:
tree, margin, feats = trees[(SEEDS[0], "Three-feature gate")]
print("margin:", margin)
print(export_text(tree, feature_names=feats, decimals=3))
importance = pd.DataFrame([dict(zip(FEATURES, trees[(s, "Three-feature gate")][0].feature_importances_)) for s in SEEDS], index=SEEDS)
importance.round(2)

margin: 0.05
|--- concentration <= 0.578
|   |--- entropy <= 1.209
|   |   |--- value: [0.004]
|   |--- entropy >  1.209
|   |   |--- value: [-0.244]
|--- concentration >  0.578
|   |--- concentration <= 0.844
|   |   |--- value: [-0.493]
|   |--- concentration >  0.844
|   |   |--- value: [0.078]



,concentration,entropy,displacement
241023,0.98,0.02,0.00
241024,0.87,0.11,0.02
241025,0.89,0.11,0.00
241026,0.94,0.00,0.06
241027,0.96,0.00,0.04
